# Borrar un usuario

## Qué hace

Borra el usuario Ada. La respuesta no trae datos: si todo va bien, se ve solo el estado `204`.

## Qué pasa en la base de datos

Quita la fila de Ada. No cambia ninguna otra fila.

## Qué acción usa

`DELETE /users/:id` con el id de Ada. El id se busca antes en `GET /users`, por el correo `ada@example.com`.

## Anotaciones

Si no hay usuario con ese id, se ve `User not found`. Si Ada no está en la lista, no hay id y se ve `Id is invalid`. Si no se puede borrar, se ve `Could not delete the user`.

Se abre con `pnpm jupyter-node-http` desde la app.

In [ ]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "src" / "logic").is_dir():
    app = app.parent
if not (app / "src" / "logic").is_dir():
    raise SystemExit("No encuentro la app.")

code = r"""
const origin = "http://127.0.0.1:3004";
const usersResponse = await fetch(origin + "/users");
const users = await usersResponse.json();
let adaId;
for (const user of users) {
  if (user.email === "ada@example.com") {
    adaId = user.id;
  }
}
const response = await fetch(origin + "/users/" + adaId, { method: "DELETE" });
if (response.ok === false) {
  const result = await response.json();
  console.log(result.message);
} else {
  console.log(response.status);
}
"""

env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")